# AO-LoRA v1 — Attribution-Orthogonal LoRA (LLaMA-3.2 3B + MultiNLI)

**Relationship to prior work:** this is a new experimental branch, not a continuation of
`siyam/esmca-v8-llama-3-2-3b-multinli-1.ipynb`. It reuses v8's backbone, LoRA injection,
and MultiNLI task setup so results are directly comparable, but replaces the routing /
forgetting-detection mechanism entirely.

**Why:** across v1-v8, the same failure kept recurring: attribution maps computed through
the *frozen* backbone are nearly identical across tasks (proto off-diagonal cosine 0.96-0.99
in every version, RoBERTa or LLaMA) because Integrated Gradients keeps highlighting generic
function words ("to", "from", "can") instead of task-discriminating content. v6's one success
(contrastive attribution, ESMCA=0.579 vs uniform=0.381 on CLINC150) and v5's diagnostic
(banking-vs-travel attribution correlation flips from **+0.96 (frozen)** to **-0.40 (adapted,
i.e. computed through the trained LoRA adapter)**) both point at the same fix: attribution
must be computed *through the task-specific adapter*, not the shared frozen backbone. v7
attempted this but mixed frozen-IG prototypes with adapted-IG queries inconsistently and
regressed. v8 never got here at all (killed by a runtime issue in the *frozen* multi-step IG
router, unrelated to this fix).

**The idea — Attribution-Orthogonal LoRA (AO-LoRA):**

1. **Consistent adapted attribution everywhere.** Every attribution map (training,
   prototypes, inference queries) is computed with *that task's own adapter active* — never
   the frozen backbone. No frozen/adapted space-mixing bug possible by construction.
2. **Cheap single-pass attribution.** Instead of multi-step Integrated Gradients (the
   compute blow-up that killed v8's evaluation — one captum IG call already expands into
   `batch_size x ig_steps` forward/backward passes through 28 LLaMA layers), we use
   **Gradient x Input** (IG's own single-step degenerate case): one forward pass, one
   backward pass, per (batch, adapter). This is the same computation v8's
   `attribute_differentiable` already implemented for its abandoned consolidation step -
   here it becomes the *primary* mechanism, used consistently.
3. **Orthogonality as a training objective, not a post-hoc hope.** O-LoRA enforces
   orthogonal adapter *subspaces in weight space*. CABLE routes by *gradient similarity*.
   Neither makes attribution-space separation an explicit training target. Here, while
   training task T+1's adapter, we add a contrastive hinge loss that directly penalizes its
   attribution vector for being too similar (cosine > margin) to any earlier task's stored
   attribution prototype:

   ```
   L_total = L_task + beta * mean_t{ relu( cos(phi_{T+1}(x), Phi_t) - margin ) }
   ```

   This turns v5's one-off diagnostic observation into a training-time guarantee instead of
   an inference-time hack.
4. **Routing without a shortlist.** Because attribution is now a single cheap forward+backward
   pass (no multi-step integration), it is affordable to compute it once per candidate adapter
   for a whole batch (T passes per batch, not per example) and compare each to its own
   prototype directly - no two-stage shortlisting needed, and no more `ig_steps`-driven
   blow-up.
5. **Forgetting is structurally near-zero here, on purpose.** Per-task adapters and per-task
   heads are frozen once trained and never touched again (matches the BWT=0.0 finding from
   v5 onward) - there is nothing left in this architecture that could "forget" in the
   classical sense. The real open problem this experiment targets is *routing without task
   labels*, and the **Attribution Separation Score (ASS)** below - mean pairwise cosine
   similarity across all stored prototypes, tracked after every task - is the metric that
   replaces the old drift-based ADS for this purpose.

**Honest limitation:** routing cost grows linearly with the number of tasks T (one
forward+backward pass per candidate adapter per batch). Fine for T <= 10-20; would need
hierarchical prototype clustering for much larger task counts. Also, Gradient x Input trades
some of IG's axiomatic guarantees (completeness) for a ~ig_steps-fold speedup - noted as a
deliberate engineering tradeoff, not an oversight.

**Before running:**
1. Accept license at huggingface.co/meta-llama/Llama-3.2-3B
2. Kaggle -> Settings -> Secrets -> add `HF_TOKEN`

In [ ]:
!pip install -q "datasets>=2.19.0" "bitsandbytes>=0.43.0" "accelerate>=0.27.0" 

In [ ]:
import os, json, random, logging, time, warnings
from dataclasses import dataclass
from typing import Dict, List, Optional
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from scipy.stats import spearmanr
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, confusion_matrix
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from huggingface_hub.utils import logging as hf_logging
from datasets.utils import logging as ds_logging
from transformers.utils import logging as tf_logging
hf_logging.set_verbosity_error(); ds_logging.set_verbosity_error(); tf_logging.set_verbosity_error()
logging.getLogger("httpx").setLevel(logging.WARNING)
warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger("ao_lora")
NUM_CLASSES = 3

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN", None)
if HF_TOKEN is None:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded from Kaggle secrets.")
    except Exception:
        print("WARNING: HF_TOKEN not found. Add it in Kaggle Secrets.")

CONFIG = {
    "backbone":            "meta-llama/Llama-3.2-3B",
    "max_seq_len":         128,
    "lora_rank":           8,
    "lora_alpha":          16,
    "genres": ["fiction", "government", "slate", "telephone", "travel"],
    "train_per_genre":     200,
    "batch_size":          4,
    "epochs_per_task":     5,
    "lr":                  2e-4,
    "n_prototype_samples": 64,
    "router_tau":          0.1,
    # AO-LoRA specific
    "ortho_margin":        0.3,   # attribution cosine similarity must stay below this vs. every past task
    "ortho_weight":        1.0,   # weight (beta) on the orthogonality loss term
    "ortho_steps_per_epoch": 1,   # how many orthogonality-loss updates per epoch (cost control)
    "seed":                42,
    "output_dir": "/kaggle/working" if os.path.isdir("/kaggle/working") else "outputs",
}

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
os.makedirs(CONFIG["output_dir"], exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 2. Load LLaMA-3.2 3B (4-bit NF4)

Unchanged from v8: model ~2.5GB + LoRA ~0.3GB + heads ~0.1GB + activations ~5GB (Gradient x
Input needs far less activation memory than multi-step IG since there is no `ig_steps`-fold
batch replication) - fits a T4 comfortably.

In [ ]:
def load_backbone(model_name, hf_token=None):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    backbone = AutoModel.from_pretrained(
        model_name, quantization_config=bnb_config,
        device_map={"": 0}, torch_dtype=torch.bfloat16,
        token=hf_token, trust_remote_code=True,
    )
    for param in backbone.parameters():
        param.requires_grad = False
    backbone.eval()
    torch.cuda.empty_cache()
    used  = torch.cuda.memory_allocated()/1e9
    total = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f"Backbone loaded. VRAM: {used:.1f}/{total:.1f} GB")
    return backbone, tokenizer


def mean_pool(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    return (hidden_states * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)


class ClassificationHead(nn.Module):
    def __init__(self, hidden, n_classes=NUM_CLASSES):
        super().__init__()
        self.dense    = nn.Linear(hidden, hidden)
        self.dropout  = nn.Dropout(0.1)
        self.out_proj = nn.Linear(hidden, n_classes)
        self.act      = nn.Tanh()
    def forward(self, x):
        return self.out_proj(self.dropout(self.act(self.dense(x))))

## 3. LoRA adapter bank (LLaMA targets: q_proj + v_proj)

Identical to v8 - the adapter mechanism itself is not what was broken.

In [ ]:
class LoRAPair(nn.Module):
    def __init__(self, d_in, d_out, rank):
        super().__init__()
        self.A = nn.Parameter(torch.empty(rank, d_in, dtype=torch.bfloat16))
        self.B = nn.Parameter(torch.zeros(d_out, rank, dtype=torch.bfloat16))
        nn.init.kaiming_uniform_(self.A, a=5**0.5)
    def forward(self, x): return (x @ self.A.t()) @ self.B.t()


class LoRAInjectedLinear(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        self.scaling = alpha / rank
        self.rank = rank
        self.in_features  = base.in_features
        self.out_features = base.out_features
        self.adapters        = nn.ModuleDict()
        self.active_task     = None
        self.routing_weights = None

    def add_task(self, name):
        adapter = LoRAPair(self.in_features, self.out_features, self.rank)
        device = next(self.base.parameters()).device
        adapter = adapter.to(device)
        self.adapters[name] = adapter
    def freeze_task(self, name):
        for p in self.adapters[name].parameters(): p.requires_grad = False
    def set_active_task(self, name):   self.active_task, self.routing_weights = name, None
    def set_routing_weights(self, w):  self.routing_weights, self.active_task = w, None

    def forward(self, x):
        base_device = self.base.weight.device
        if x.device != base_device:
            x = x.to(base_device)
        out = self.base(x)
        if self.active_task and self.active_task in self.adapters:
            adapter = self.adapters[self.active_task]
            if next(adapter.parameters()).device != base_device:
                adapter.to(base_device)
            out = out + self.scaling * adapter(x)
        elif self.routing_weights:
            for name, w in self.routing_weights.items():
                if w and name in self.adapters:
                    adapter = self.adapters[name]
                    if next(adapter.parameters()).device != base_device:
                        adapter.to(base_device)
                    out = out + w * self.scaling * adapter(x)
        return out


class AdapterBank:
    def __init__(self, injected):
        self.injected, self.task_order = injected, []
    def add_task(self, name):
        self.task_order.append(name)
        for _, m in self.injected: m.add_task(name)
    def freeze_task(self, name):
        for _, m in self.injected: m.freeze_task(name)
    def set_active_task(self, name):
        for _, m in self.injected: m.set_active_task(name)
    def set_routing_weights(self, w):
        for _, m in self.injected: m.set_routing_weights(w)
    def trainable_parameters(self, name):
        for _, m in self.injected: yield from m.adapters[name].parameters()


def inject_lora(backbone, rank, alpha):
    injected = []
    for li, layer in enumerate(backbone.layers):
        for attr in ("q_proj", "v_proj"):
            base = getattr(layer.self_attn, attr)
            wrapped = LoRAInjectedLinear(base, rank, alpha)
            setattr(layer.self_attn, attr, wrapped)
            injected.append((f"layer{li}.{attr}", wrapped))
    logger.info(f"LoRA injected: {len(injected)} projections ({len(backbone.layers)} layers x 2)")
    return AdapterBank(injected)

## 4. AOLoRAModel (LLaMA — mean pooling, embed_tokens, per-task heads)

Same shape as v8's `ESMCAModel`. `forward_frozen`/`probe_head` are kept only for the sanity
gate (checking the raw backbone + adapters behave sanely) — they play **no role** in routing
or prototypes anymore, which is the whole point: no frozen-space signal is used for anything
that needs to discriminate between tasks.

In [ ]:
class AOLoRAModel(nn.Module):
    def __init__(self, backbone_name, lora_rank, lora_alpha, hf_token=None):
        super().__init__()
        self.backbone, self.tokenizer = load_backbone(backbone_name, hf_token)
        self.adapter_bank = inject_lora(self.backbone, lora_rank, lora_alpha)
        self.heads = nn.ModuleDict()
        self.probe_head: Optional[ClassificationHead] = None
        self._hidden = self.backbone.config.hidden_size

    def _pool(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        return mean_pool(out.last_hidden_state, attention_mask)

    def forward_task(self, input_ids, attention_mask, task):
        self.adapter_bank.set_active_task(task)
        return self.heads[task](self._pool(input_ids, attention_mask))

    def forward_composed(self, input_ids, attention_mask, weights):
        self.adapter_bank.set_routing_weights(weights)
        pooled = self._pool(input_ids, attention_mask)
        self.adapter_bank.set_routing_weights(None)
        logits = None
        for t, w in weights.items():
            if w and t in self.heads:
                l = self.heads[t](pooled) * w
                logits = l if logits is None else logits + l
        return logits

    def forward_frozen(self, input_ids, attention_mask):
        self.adapter_bank.set_active_task(None)
        return self.probe_head(self._pool(input_ids, attention_mask))

    def start_new_task(self, name):
        self.adapter_bank.add_task(name)
        dev = next(p for p in self.backbone.parameters() if p.device.type == "cuda").device
        self.heads[name] = ClassificationHead(self._hidden).to(dev)
        if self.probe_head is None:
            self.probe_head = ClassificationHead(self._hidden).to(dev)
        for _, m in self.adapter_bank.injected:
            if name in m.adapters:
                m.adapters[name] = m.adapters[name].to(dev)
        self.adapter_bank.set_active_task(name)

    def freeze_task(self, name):
        self.adapter_bank.freeze_task(name)
        for p in self.heads[name].parameters(): p.requires_grad = False

    def task_trainable_parameters(self, name):
        yield from self.adapter_bank.trainable_parameters(name)
        yield from self.heads[name].parameters()

## 5. Adapted Attribution Extractor (AAE) — single mechanism, used everywhere

The one component that is genuinely different from every prior version. Computes
**Gradient x Input** (single forward + single backward pass, no multi-step integration)
*with a specific task's adapter active*. The same function is used for:

- prototype computation (`create_graph=False`, `.detach()`ed)
- inference routing queries (`create_graph=False`)
- the training-time orthogonality loss (`create_graph=True`, so gradients flow back into
  the adapter currently being trained)

Using one function for all three eliminates the frozen/adapted space-mixing bug that broke
v7, and removes the multi-step IG cost that made v8's evaluation infeasible on Kaggle's
12-hour limit.

In [ ]:
class AdaptedAttributionExtractor:
    def __init__(self, model: AOLoRAModel):
        self.model = model

    def _grad_x_input(self, input_ids, attention_mask, task_name, create_graph):
        self.model.adapter_bank.set_active_task(task_name)
        embed_fn = self.model.backbone.embed_tokens
        embeds = embed_fn(input_ids).detach().float().requires_grad_(True)
        out = self.model.backbone(inputs_embeds=embeds.to(torch.bfloat16),
                                   attention_mask=attention_mask)
        pooled = mean_pool(out.last_hidden_state.float(), attention_mask)
        logits = self.model.heads[task_name](pooled)
        target = logits.argmax(-1).detach()
        selected = logits.gather(1, target.unsqueeze(1)).squeeze(1).sum()
        (grads,) = torch.autograd.grad(selected, embeds, create_graph=create_graph)
        scores = ((embeds * grads).sum(dim=-1) * attention_mask).abs()
        with torch.no_grad():
            E = embed_fn(input_ids).float()
        w = scores / (scores.sum(dim=1, keepdim=True) + 1e-8)
        attr = torch.einsum("bl,bld->bd", w, E)
        return F.normalize(attr, dim=-1)

    def attribute(self, input_ids, attention_mask, task_name):
        '''Non-differentiable query/prototype attribution (batched).'''
        with torch.enable_grad():
            return self._grad_x_input(input_ids, attention_mask, task_name, create_graph=False).detach()

    def attribute_for_training(self, input_ids, attention_mask, task_name):
        '''Differentiable attribution -- gradients flow back into task_name adapter.'''
        return self._grad_x_input(input_ids, attention_mask, task_name, create_graph=True)

    def compute_prototype(self, loader, task_name, n_samples, device):
        collected, seen = [], 0
        for batch in loader:
            if seen >= n_samples: break
            batch = {k: v.to(device) for k, v in batch.items()}
            a = self.attribute(batch["input_ids"], batch["attention_mask"], task_name)
            collected.append(a.cpu()); seen += a.shape[0]
        return F.normalize(torch.cat(collected)[:n_samples].mean(0), dim=-1)

## 6. Attribution-Orthogonal Router + Orthogonality Loss

**Router:** for a batch of unlabeled inputs, compute attribution under *every* stored
adapter (T forward+backward passes for the whole batch — not per example, and not
multiplied by `ig_steps` — this is the fix for v8's runtime blow-up), compare each to its
own prototype, softmax over tasks per example.

**Orthogonality loss:** the direct extension of O-LoRA's weight-space orthogonality into
attribution space — a contrastive hinge that penalizes the new task's attribution for
being too similar (cosine > margin) to any earlier task's stored prototype. Optimized
*during* training, not checked after the fact.

In [ ]:
class AttributionOrthogonalRouter:
    def __init__(self, tau=0.1):
        self.tau = tau

    def route(self, input_ids, attention_mask, extractor: AdaptedAttributionExtractor, prototypes: Dict[str, torch.Tensor]):
        names = list(prototypes.keys())
        sims = []
        for name in names:
            a = extractor.attribute(input_ids, attention_mask, name)          # (B, d)
            proto = prototypes[name].to(a.device).unsqueeze(0)                # (1, d)
            sims.append(F.cosine_similarity(a, proto, dim=-1))                # (B,)
        S = torch.stack(sims, dim=1)                                          # (B, T)
        W = F.softmax(S / self.tau, dim=1)
        return names, W


class AttributionOrthogonalityLoss:
    def __init__(self, margin=0.3, weight=1.0):
        self.margin, self.weight = margin, weight

    def __call__(self, current_attr: torch.Tensor, past_prototypes: Dict[str, torch.Tensor]):
        if not past_prototypes:
            return torch.tensor(0.0, device=current_attr.device)
        terms = []
        for proto in past_prototypes.values():
            proto = proto.to(current_attr.device)
            cos = F.cosine_similarity(current_attr.unsqueeze(0), proto.unsqueeze(0)).squeeze(0)
            terms.append(F.relu(cos - self.margin))
        return self.weight * torch.stack(terms).mean()

## 7. MultiNLI — 5 matched genres as continual tasks

Unchanged from v8: premise + [SEP] + hypothesis, 3-way NLI, `val_matched` for proto/eval,
`val_mismatched` for cross-genre generalization. Kept identical so results are comparable.

In [ ]:
@dataclass
class Task:
    name: str
    train_loader: DataLoader
    proto_loader: DataLoader
    eval_loader:  DataLoader
    mismatch_loader: Optional[DataLoader] = None


class MultiNLIDataset(Dataset):
    def __init__(self, examples, tokenizer, max_seq_len):
        self.examples, self.tok, self.L = examples, tokenizer, max_seq_len
    def __len__(self): return len(self.examples)
    def __getitem__(self, i):
        ex  = self.examples[i]
        txt = ex["premise"] + " [SEP] " + ex["hypothesis"]
        enc = self.tok(txt, truncation=True, max_length=self.L,
                       padding="max_length", return_tensors="pt")
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(ex["label"], dtype=torch.long),
        }


def load_multinli_tasks(tokenizer, max_seq_len, batch_size,
                        genres=None, train_per_genre=200, seed=42):
    if genres is None:
        genres = ["fiction","government","slate","telephone","travel"]
    raw = load_dataset("nyu-mll/multi_nli")
    rng = np.random.RandomState(seed)
    train_all = raw["train"].filter(lambda x: x["label"] != -1)
    val_m     = raw["validation_matched"].filter(lambda x: x["label"] != -1)
    val_mm    = raw["validation_mismatched"].filter(lambda x: x["label"] != -1)
    tasks = []
    for genre in genres:
        tr = [ex for ex in train_all if ex["genre"] == genre]
        per_label = train_per_genre // 3
        tr_ex = []
        for lbl in range(3):
            sub = [ex for ex in tr if ex["label"] == lbl]
            chosen = rng.permutation(len(sub))[:per_label]
            tr_ex += [sub[i] for i in chosen]
        rng.shuffle(tr_ex)
        va  = [ex for ex in val_m  if ex["genre"] == genre]
        mm  = [ex for ex in val_mm if ex["genre"] == genre]
        cut = len(va)//2
        proto_ex, eval_ex = va[:cut], va[cut:]
        mk = lambda exs, sh: DataLoader(MultiNLIDataset(exs, tokenizer, max_seq_len),
                                        batch_size=batch_size, shuffle=sh, drop_last=False)
        tasks.append(Task(genre, mk(tr_ex,True), mk(proto_ex,False),
                          mk(eval_ex,False), mk(mm,False) if mm else None))
        logger.info(f"{genre:12s} train={len(tr_ex)} proto={len(proto_ex)} eval={len(eval_ex)} mismatch={len(mm)}")
    return tasks

## 8. ContinualTrainer (AO-LoRA)

Same overall shape as v8's trainer, but the per-task loop now includes an
**orthogonality step**: once per epoch, pull one batch of the new task's own training data,
compute its (differentiable) attribution through the adapter currently being trained, and
push it away from every previously stored prototype via the hinge loss above. This replaces
v8's after-the-fact drift check + consolidation — separation is trained in, not inspected
after training and patched.

In [ ]:
class ContinualTrainer:
    def __init__(self, model, cfg):
        self.model  = model
        self.cfg    = cfg
        self.attributor = AdaptedAttributionExtractor(model)
        self.router     = AttributionOrthogonalRouter(tau=cfg["router_tau"])
        self.ortho_loss = AttributionOrthogonalityLoss(margin=cfg["ortho_margin"], weight=cfg["ortho_weight"])
        self.prototypes: Dict[str, torch.Tensor] = {}
        self.tasks:      Dict[str, Task]         = {}
        self.separation_history: List[dict] = []
        self.device = next(p for p in model.backbone.parameters() if p.device.type == "cuda").device

    def _to(self, b): return {k: v.to(self.device) for k, v in b.items()}

    def _orthogonality_step(self, optimizer, task_name):
        if not self.prototypes:
            return None
        b = self._to(next(iter(self.tasks[task_name].train_loader)))
        optimizer.zero_grad()
        attr = self.attributor.attribute_for_training(b["input_ids"], b["attention_mask"], task_name)
        loss = self.ortho_loss(attr.mean(0), self.prototypes)
        if loss.item() > 0:
            loss.backward()
            optimizer.step()
        return float(loss.item())

    def train_task(self, task):
        name = task.name
        self.tasks[name] = task
        self.model.start_new_task(name)
        torch.cuda.empty_cache()
        optimizer = AdamW(list(self.model.task_trainable_parameters(name)), lr=self.cfg["lr"])
        for ep in range(self.cfg["epochs_per_task"]):
            self.model.backbone.eval()
            for h in self.model.heads.values(): h.train()
            if self.model.probe_head: self.model.probe_head.train()
            bar = tqdm(task.train_loader,
                       desc=f"[{name}] ep {ep+1}/{self.cfg['epochs_per_task']}", leave=False)
            for b in bar:
                b = self._to(b)
                optimizer.zero_grad()
                loss = F.cross_entropy(
                    self.model.forward_task(b["input_ids"], b["attention_mask"], name),
                    b["labels"])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    list(self.model.task_trainable_parameters(name)), 1.0)
                optimizer.step()
                bar.set_postfix(loss=f"{loss.item():.3f}")
            for _ in range(self.cfg["ortho_steps_per_epoch"]):
                ortho = self._orthogonality_step(optimizer, name)
            if ortho is not None:
                logger.info(f"[{name}] ep {ep+1} orthogonality loss: {ortho:.4f}")
        self.model.backbone.eval()
        torch.cuda.empty_cache()
        proto = self.attributor.compute_prototype(task.proto_loader, name, self.cfg["n_prototype_samples"], self.device)
        self.prototypes[name] = proto.cpu()
        self.model.freeze_task(name)
        if len(self.prototypes) > 1:
            names = list(self.prototypes.keys())
            P = torch.stack([self.prototypes[n] for n in names])
            S = (P @ P.t()).numpy()
            off = S[~np.eye(len(names), dtype=bool)]
            self.separation_history.append({"after_task": name, "mean_offdiag": float(off.mean()), "max_offdiag": float(off.max())})
            logger.info(f"Done '{name}'. Proto off-diag: mean={off.mean():.3f} max={off.max():.3f}")
        else:
            logger.info(f"Done '{name}'.")

## 9. Three inference modes

In [ ]:
@torch.no_grad()
def eval_oracle(trainer, task):
    trainer.model.backbone.eval()
    correct = total = 0
    for b in task.eval_loader:
        b = trainer._to(b)
        preds = trainer.model.forward_task(b["input_ids"], b["attention_mask"], task.name).argmax(-1)
        correct += (preds == b["labels"]).sum().item(); total += len(preds)
    return correct/total if total else 0.0


@torch.no_grad()
def eval_uniform(trainer, task):
    trainer.model.backbone.eval()
    names = list(trainer.prototypes.keys())
    w = {t: 1.0/len(names) for t in names}
    correct = total = 0
    for b in task.eval_loader:
        b = trainer._to(b)
        preds = trainer.model.forward_composed(b["input_ids"], b["attention_mask"], w).argmax(-1)
        correct += (preds == b["labels"]).sum().item(); total += len(preds)
    return correct/total if total else 0.0


def eval_ao_lora(trainer, task, true_task_idx, task_order, loader=None):
    '''Routing (T forward+backward passes per batch, via attribution) needs grad enabled;
    the final blended prediction does not.'''
    trainer.model.backbone.eval()
    if loader is None: loader = task.eval_loader
    correct = total = 0
    routed_idx, true_idx = [], []
    for b in tqdm(loader, desc=f"eval {task.name}", leave=False):
        b = trainer._to(b)
        with torch.enable_grad():
            names, W = trainer.router.route(b["input_ids"], b["attention_mask"],
                                             trainer.attributor, trainer.prototypes)
        W = W.detach()
        with torch.no_grad():
            for i in range(b["input_ids"].shape[0]):
                w = {n: float(W[i, j]) for j, n in enumerate(names)}
                routed_idx.append(task_order.index(max(w, key=w.get)))
                true_idx.append(true_task_idx)
                logits = trainer.model.forward_composed(
                    b["input_ids"][i:i+1], b["attention_mask"][i:i+1], w)
                correct += int(logits.argmax(-1).item() == b["labels"][i].item())
                total   += 1
    return (correct/total if total else 0.0), routed_idx, true_idx

## 10. Metrics + separability

`attribution_separation_score` replaces the old drift-based ADS: since per-task
adapters/heads are frozen and never revisited, there is no parameter drift to measure here
by construction (see the BWT=0.0 finding from v5 onward in the report) — the metric that
actually matters is whether the *stored* prototypes stay separated as more tasks accumulate,
which is exactly what the orthogonality loss optimizes for during training.

In [ ]:
def compute_acc(R): return float(np.mean(R[-1]))
def compute_bwt(R):
    T=len(R); return 0.0 if T<2 else float(np.mean([R[T-1][t]-R[t][t] for t in range(T-1)]))
def compute_fwt(R):
    T=len(R); return 0.0 if T<2 else float(np.mean([R[i-1][i] for i in range(1,T)]))
def ris(true_idx, routed_idx):
    if len(true_idx)<2: return 0.0
    c,_=spearmanr(true_idx,routed_idx); return float(c) if c==c else 0.0
def attribution_separation_score(prototypes: Dict[str, torch.Tensor]):
    names = list(prototypes.keys())
    if len(names) < 2: return 0.0
    P = torch.stack([prototypes[n] for n in names])
    S = (P @ P.t()).numpy()
    off = S[~np.eye(len(names), dtype=bool)]
    return float(off.mean())

def collect_sample_attributions(trainer, tasks, per_task=40):
    '''Offline diagnostic: each sample attribution computed through its OWN true task
    adapter (ground truth known here, unlike at real inference) -- shows whether the
    orthogonality-trained adapters actually produce separable attribution clusters.'''
    X, y, names = [], [], [t.name for t in tasks]
    for j, task in enumerate(tasks):
        seen = 0
        for b in task.eval_loader:
            if seen >= per_task: break
            b = trainer._to(b)
            a = trainer.attributor.attribute(b["input_ids"], b["attention_mask"], task.name)
            take = min(per_task-seen, a.shape[0])
            X.append(a[:take].cpu()); y += [j]*take; seen += take
    return torch.cat(X).numpy(), np.array(y), names

def separability_report(X, y, names, out_dir):
    sil  = float(silhouette_score(X, y)) if len(names)>1 else 0.0
    perp = max(2, min(30, len(X)//4))
    emb  = TSNE(n_components=2, perplexity=perp, init="pca", random_state=0).fit_transform(X)
    plt.figure(figsize=(7,6))
    for j,n in enumerate(names):
        m=y==j; plt.scatter(emb[m,0],emb[m,1],s=18,label=n,alpha=0.8)
    plt.legend(fontsize=9)
    plt.title(f"Adapted attribution embeddings — AO-LoRA (silhouette={sil:.3f})")
    p = os.path.join(out_dir, "attribution_tsne_ao_lora.png")
    plt.tight_layout(); plt.savefig(p, dpi=150); plt.close()
    print(f"t-SNE saved -> {p}")
    return sil

## 11. Sanity gate

Checks (a)/(b) are the same basic adapter-health checks as v8. Check (c) is new: it
compares Gradient x Input attribution between two freshly-probed adapters *before* any
orthogonality training has happened, purely to confirm the extractor runs correctly and
produces finite, non-degenerate vectors. The real proof of separation only appears after
>=2 full tasks have been trained with the orthogonality loss (see the separation history
logged during Section 12).

In [ ]:
model   = AOLoRAModel(CONFIG["backbone"], CONFIG["lora_rank"], CONFIG["lora_alpha"], hf_token=HF_TOKEN)
tasks   = load_multinli_tasks(model.tokenizer, CONFIG["max_seq_len"], CONFIG["batch_size"],
                               CONFIG["genres"], CONFIG["train_per_genre"], CONFIG["seed"])
trainer = ContinualTrainer(model, CONFIG)

print(f"\nVRAM after load: {torch.cuda.memory_allocated()/1e9:.1f} / "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

_b = trainer._to(next(iter(tasks[0].train_loader)))
trainer.model.start_new_task("_probe")
torch.cuda.empty_cache()
_opt = AdamW(list(trainer.model.task_trainable_parameters("_probe")), lr=CONFIG["lr"])
trainer.model.backbone.eval(); trainer.model.heads["_probe"].train()
for step in range(40):
    _opt.zero_grad()
    _loss = F.cross_entropy(
        trainer.model.forward_task(_b["input_ids"], _b["attention_mask"], "_probe"), _b["labels"])
    _loss.backward()
    torch.nn.utils.clip_grad_norm_(list(trainer.model.task_trainable_parameters("_probe")), 1.0)
    _opt.step()
trainer.model.backbone.eval()
with torch.no_grad():
    _on  = trainer.model.forward_task(_b["input_ids"], _b["attention_mask"], "_probe")
    trainer.model.adapter_bank.set_active_task(None)
    _off = trainer.model.probe_head(
        mean_pool(trainer.model.backbone(_b["input_ids"], _b["attention_mask"]).last_hidden_state.float(),
                  _b["attention_mask"]))
chance = float(np.log(NUM_CLASSES))
print(f"(a) probe loss after 40 steps: {_loss.item():.3f}  (chance={chance:.3f})")
print(f"(b) adapter max|delta logit|: {(_on.float()-_off.float()).abs().max().item():.3f}")
assert _loss.item() < 0.7 * chance, "SANITY FAIL (a): loss stuck — check LR or HF_TOKEN"
assert (_on.float()-_off.float()).abs().max().item() > 0.1, "SANITY FAIL (b): adapters inert"

with torch.no_grad():
    b0 = trainer._to(next(iter(tasks[0].eval_loader)))
    a0 = trainer.attributor.attribute(b0["input_ids"], b0["attention_mask"], "_probe").mean(0)
print(f"(c) adapted Gradient x Input attribution vector: finite={torch.isfinite(a0).all().item()}, "
      f"norm={a0.norm().item():.3f}")
assert torch.isfinite(a0).all(), "SANITY FAIL (c): attribution extractor produced NaN/Inf"

trainer.model.adapter_bank.set_active_task(None)
del trainer.model.heads["_probe"]
for _, m in trainer.model.adapter_bank.injected:
    del m.adapters["_probe"]
trainer.model.adapter_bank.task_order.remove("_probe")
print("\nSanity gate PASSED -- safe to run full training.")
torch.cuda.empty_cache()

## 12. Sequential training

Watch the "Proto off-diag" line logged after each task from the 2nd task onward -- this is
the direct, per-task readout of whether the orthogonality loss is doing its job. It should
stay well below the frozen-IG baseline of 0.96-0.99 seen in every prior version.

In [ ]:
t_start = time.time()
for i, task in enumerate(tasks):
    trainer.train_task(task)
    elapsed   = (time.time()-t_start)/60
    remaining = elapsed/(i+1)*(len(tasks)-i-1)
    logger.info(f"[{i+1}/{len(tasks)}] elapsed {elapsed:.1f} min | est. remaining ~{remaining:.0f} min")
    torch.cuda.empty_cache()
print(f"\nPhase 1 complete in {(time.time()-t_start)/60:.1f} min.")

## 13. R-matrix

Routing here costs `T` (adapter count) single-pass forward+backward calls per batch — no
`ig_steps` multiplier — so this should complete in minutes, not the ~9.5 hours per row that
killed v8.

In [ ]:
task_order = trainer.model.adapter_bank.task_order
R = []; row_times = []; t_start = time.time()
for i, task in enumerate(tasks):
    t_row = time.time(); row = []
    for j, et in enumerate(tasks):
        t0  = time.time()
        acc, _, _ = eval_ao_lora(trainer, et, j, task_order)
        row.append(acc)
        print(f"  eval {et.name:12s} acc={acc:.3f}  ({time.time()-t0:.0f}s)")
    R.append(row); row_times.append(time.time()-t_row)
    left = (len(tasks)-i-1)*np.mean(row_times)/60
    print(f"[{i+1}/{len(tasks)}] '{task.name}' | elapsed {(time.time()-t_start)/60:.1f} min | ~{left:.0f} min left")
    torch.cuda.empty_cache()

## 14. Final three-way comparison + cross-genre generalization

In [ ]:
oracle_accs, uniform_accs, ao_accs = [], [], []
all_true, all_routed = [], []
for j, task in enumerate(tasks):
    o = eval_oracle(trainer, task)
    u = eval_uniform(trainer, task)
    e, r_idx, t_idx = eval_ao_lora(trainer, task, j, task_order)
    oracle_accs.append(o); uniform_accs.append(u); ao_accs.append(e)
    all_routed += r_idx; all_true += t_idx
    print(f"{task.name:12s}  oracle={o:.3f}  AO-LoRA={e:.3f}  uniform={u:.3f}")

cm     = confusion_matrix(all_true, all_routed, labels=list(range(len(tasks))))
recall = np.diag(cm).astype(float)/(cm.sum(axis=1)+1e-8)
balanced_routing = float(np.mean(recall))
routing_acc = float(np.mean(np.array(all_true)==np.array(all_routed)))
print(f"\nMean:  oracle={np.mean(oracle_accs):.3f}  AO-LoRA={np.mean(ao_accs):.3f}  uniform={np.mean(uniform_accs):.3f}")
print(f"Routing accuracy:          {routing_acc:.3f}")
print(f"Balanced routing accuracy: {balanced_routing:.3f}  (chance={1/len(tasks):.3f})")
print("Per-task recall:", dict(zip(CONFIG["genres"], np.round(recall,3))))
print("Confusion matrix:"); print(cm)

print("\n=== Cross-genre generalization ===")
for j, task in enumerate(tasks):
    if task.mismatch_loader is None: continue
    e_mm,_,_ = eval_ao_lora(trainer, task, j, task_order, loader=task.mismatch_loader)
    print(f"{task.name:12s}  matched={ao_accs[j]:.3f}  mismatched={e_mm:.3f}")
torch.cuda.empty_cache()

## 15. Full results + t-SNE

In [ ]:
acc=compute_acc(R); bwt=compute_bwt(R); fwt=compute_fwt(R)
ass_score = attribution_separation_score(trainer.prototypes)
ris_score = ris(all_true,all_routed)
X,y,names = collect_sample_attributions(trainer, tasks, per_task=40)
silhouette = separability_report(X, y, names, CONFIG["output_dir"])

results = {
    "version":  "v9-ao-lora-llama3.2-3B-multinli",
    "backbone": CONFIG["backbone"],
    "dataset":  "MultiNLI (5 matched genres, NLI 3-way)",
    "config":   {k:v for k,v in CONFIG.items() if k!="output_dir"},
    "R": R, "ACC": acc, "BWT": bwt, "FWT": fwt,
    "attribution_separation_score": ass_score,
    "separation_history": trainer.separation_history,
    "RIS": ris_score,
    "routing_accuracy": routing_acc,
    "balanced_routing_accuracy": balanced_routing,
    "per_task_routing_recall": dict(zip(CONFIG["genres"], recall.tolist())),
    "confusion_matrix": cm.tolist(),
    "per_task": {t.name:{"oracle":o,"ao_lora":e,"uniform":u}
                 for t,o,e,u in zip(tasks,oracle_accs,ao_accs,uniform_accs)},
    "mean_oracle":  float(np.mean(oracle_accs)),
    "mean_ao_lora": float(np.mean(ao_accs)),
    "mean_uniform": float(np.mean(uniform_accs)),
    "sample_attribution_silhouette": silhouette,
    "task_order": task_order,
}
out = os.path.join(CONFIG["output_dir"], "ao_lora_v1_results.json")
with open(out,"w") as f: json.dump(results,f,indent=2)
print(json.dumps({k:results[k] for k in [
    "ACC","BWT","FWT","attribution_separation_score","RIS",
    "routing_accuracy","balanced_routing_accuracy",
    "mean_oracle","mean_ao_lora","mean_uniform",
    "sample_attribution_silhouette"]},indent=2))
print(f"Saved -> {out}")

## 16. Limitations / follow-ups (for the write-up)

- **Routing cost is O(T)** (one forward+backward pass per stored adapter per batch). Fine
  for the 5-task MultiNLI setup here; would need hierarchical prototype clustering or a
  cheap pre-filter for T in the hundreds.
- **Gradient x Input vs. full Integrated Gradients** trades some of IG's axiomatic
  guarantees (completeness, symmetry) for a ~`ig_steps`-fold speedup. Worth an ablation
  re-running with a few IG steps (e.g. captum's `LayerIntegratedGradients`, `n_steps=2-4`)
  restricted to the *adapted* pathway only, to see whether the orthogonality loss still
  separates prototypes and whether accuracy/routing improves further — this was never
  tried because prior versions never got a clean adapted-only pipeline running.
- **Margin/weight sensitivity**: `ortho_margin` and `ortho_weight` are untuned defaults —
  worth a small sweep before treating any single run's numbers as final.
- **Compare directly against v6's contrastive-attribution result on CLINC150** (ESMCA=0.579
  vs uniform=0.381) by re-running this exact notebook's pipeline on CLINC150 instead of
  MultiNLI — same backbone family question (does this generalize across datasets, not just
  genres of one dataset).